# Evaluation on validation.csv

In [2]:
import joblib
import pandas as pd
import numpy as np

In [3]:


best_gb = joblib.load('../weights/best_model_gb_tuned.pkl')
scaler  = joblib.load('../weights/scaler.pkl')

selected_features = ['distance', 'quote_signal', 'weight', 'market_index']

In [4]:
val_df = pd.read_csv('../dataset/validation.csv')


In [5]:
missing = val_df.isnull().sum()
missing_pct = (val_df.isnull().sum() / len(val_df)) * 100

missing_df = pd.DataFrame({
    'missing_count': missing,
    'missing_pct': missing_pct
}).sort_values('missing_pct', ascending=False)

print(missing_df)

              missing_count  missing_pct
market_index            249        2.075
weight                  165        1.375
delivery                  0        0.000
pickup                    0        0.000
load_id                   0        0.000
pickup_lon                0        0.000
pickup_lat                0        0.000
delivery_lat              0        0.000
delivery_lon              0        0.000
equipment                 0        0.000
distance                  0        0.000
date                      0        0.000
quote_signal              0        0.000


In [6]:
# Market index → average of the same date
val_df["market_index"] = val_df["market_index"].fillna(
    val_df.groupby("date")["market_index"].transform("mean")
)

# Weight → overall dataset average
val_df["weight"] = val_df["weight"].fillna(
    val_df["weight"].mean()
)

# Check missing values
print(val_df[["market_index", "weight"]].isna().sum())

market_index    0
weight          0
dtype: int64


In [23]:
X_val        = val_df[selected_features]


X_val_scaled = scaler.transform(X_val)

val_df['predicted_rate'] = best_gb.predict(X_val_scaled)

# save load_id + predicted_rate only
predictions = val_df[['load_id', 'predicted_rate']]
predictions.to_csv('../results/validation_predictions.csv', index=False)

print("Saved validation_predictions.csv:", predictions.shape)
print(predictions.head())

Saved validation_predictions.csv: (12000, 2)
     load_id  predicted_rate
0  TE-000001      767.334660
1  TE-000002     4517.637738
2  TE-000003     5661.646285
3  TE-000004     4348.360572
4  TE-000005     1733.437257


# Prediction on december_chart_inputs.csv

In [7]:
dec_df = pd.read_csv('../dataset/december-chart-inputs.csv')


In [10]:
import pandas as pd

# Load files
december = dec_df
validation = val_df

# Make sure date format is consistent
december["date"] = pd.to_datetime(december["date"])
validation["date"] = pd.to_datetime(validation["date"])


# --------------------------------------------------
# 1. Create lookup using pickup + delivery + date
# --------------------------------------------------

validation_lookup = (
    validation
    .groupby(["pickup", "delivery", "date"], as_index=False)
    [["quote_signal", "market_index"]]
    .mean()
)

# Merge into December
december = december.merge(
    validation_lookup,
    on=["pickup", "delivery", "date"],
    how="left"
)


# --------------------------------------------------
# 2. For unmatched rows, use the mean for that DATE
# --------------------------------------------------

date_mean = (
    validation
    .groupby("date")[["quote_signal", "market_index"]]
    .mean()
    .rename(columns={
        "quote_signal": "quote_signal_date_mean",
        "market_index": "market_index_date_mean"
    })
)

# Add date means
december = december.merge(
    date_mean,
    on="date",
    how="left"
)


# --------------------------------------------------
# 3. Fill missing values using date-specific mean
# --------------------------------------------------

december["market_index"] = december["market_index"].fillna(
    december["market_index_date_mean"]
)

december["quote_signal"] = december["quote_signal"].fillna(
    december["quote_signal_date_mean"]
)


# Remove helper columns
december = december.drop(
    columns=["quote_signal_date_mean", "market_index_date_mean"]
)


# --------------------------------------------------
# 4. Save completed file
# --------------------------------------------------

december.to_csv("december_completed.csv", index=False)

# Check
print("Remaining missing values:")
print(december[["quote_signal", "market_index"]].isna().sum())

Remaining missing values:
quote_signal    0
market_index    0
dtype: int64


In [11]:

# scale & predict
X_dec        = december[selected_features]
X_dec_scaled = scaler.transform(X_dec)
december['predicted_rate'] = best_gb.predict(X_dec_scaled)

# keep only required columns in exact order
december_output = december[['pickup','delivery','distance','equipment','weight','date','predicted_rate']]
december_output.to_csv('../data/december_chart_inputs.csv', index=False)

print("Saved december_chart_inputs.csv")
print(december_output)

Saved december_chart_inputs.csv
       pickup    delivery  distance equipment  weight       date  \
0   Lexington  Fort Wayne       360   Dry Van   32000 2025-12-01   
1   Lexington  Fort Wayne       360   Dry Van   32000 2025-12-02   
2   Lexington  Fort Wayne       360   Dry Van   32000 2025-12-03   
3   Lexington  Fort Wayne       360   Dry Van   32000 2025-12-04   
4   Lexington  Fort Wayne       360   Dry Van   32000 2025-12-05   
5   Lexington  Fort Wayne       360   Dry Van   32000 2025-12-06   
6   Lexington  Fort Wayne       360   Dry Van   32000 2025-12-07   
7   Lexington  Fort Wayne       360   Dry Van   32000 2025-12-08   
8   Lexington  Fort Wayne       360   Dry Van   32000 2025-12-09   
9   Lexington  Fort Wayne       360   Dry Van   32000 2025-12-10   
10  Lexington  Fort Wayne       360   Dry Van   32000 2025-12-11   
11  Lexington  Fort Wayne       360   Dry Van   32000 2025-12-12   
12  Lexington  Fort Wayne       360   Dry Van   32000 2025-12-13   
13  Lexington  F